# Tutorial 3.1: Thermoelastic Topology Optimisation (TO)

In this tutorial, we will explore Topology Optimisation (TO) applied to a structure experiencing both mechanical loads and thermal expansion. We will run the optimisation through four distinct phases to highlight several critical takeaways in thermoelastic design:

1. **Elastic TO is simple and well-defined:** When dealing with purely mechanical loads, reducing compliance (maximising stiffness) inherently tends to reduce overall stresses.
2. **Thermoelastic TO (Compliance Objective) can be misleading:** When optimising for compliance under thermal loads, the optimiser will often exploit thermal expansion to counteract mechanical displacement. While this minimises the objective, it completely ignores the internal thermal stresses generated, often leading to unviable designs.
3. **Stress constraints are necessary but tricky:** Adding stress constraints to a thermoelastic compliance problem yields much safer designs. However, standard volume constraints often become ill-defined or inactive because the "thermal load" itself is dependent on the design (more material = more thermal force).
4. **Volume Minimisation is often the most meaningful approach:** For thermoelastic problems, setting the objective to minimise volume while explicitly constraining maximum stress and compliance provides a much more robust and well-posed formulation.

## 1. Standard Imports and Setup

First, we import Firedrake, the `adjoint` module, and our optimisation and visualisation libraries. MMA can be imported from the GitHub repository.

In [ ]:
try:
    from firedrake import *
    from firedrake.adjoint import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *
    from firedrake.adjoint import *

import numpy as np
import matplotlib.pyplot as plt

try:
    from mma import gcmmasub, asymp, concheck, raaupdate
except ImportError:
    !wget https://raw.githubusercontent.com/Jean-LucLeonetti/ukhdtn-tutorials/main/mma.py
    from mma import gcmmasub, asymp, concheck, raaupdate

## 2. Domain, Mesh, and Material Setup

We define a rectangular domain pinned at the left and right edges. We apply a downward mechanical traction on the top boundary. To ensure the load can always be applied, we define a thin "non-design" layer of solid material along the top edge that the optimiser is not allowed to remove.

In [ ]:
# --- DOMAIN & MESH ---
Lx, Ly = 3.0, 1.0        # Domain dimensions
nx, ny = 150, 50         # Number of elements

# --- MATERIAL PROPERTIES ---
E_min = 1e-6             # Young's modulus of void
E_max = 1.0              # Young's modulus of solid material
nu = 0.3                 # Poisson's ratio
p_mat = 5.0              # SIMP penalty parameter
alpha_val = 1.           # Coefficient of Thermal Expansion (CTE)
delta_T_target = 50.0    # Uniform temperature change for Phase 2 & 3

# --- LOADS & BCs ---
traction_y = -2.0        # Downward traction magnitude on top boundary

# --- STRESS CONSTRAINT ---
sigma_yield_val = 80.0   # Yield stress limit
c_penalty_val = 100.0    # Quadratic penalty multiplier for Phase 3

# --- OPTIMIZATION & FILTER ---
vol_frac = 0.3           # Target volume fraction
r_min_val = 0.05         # Filter radius
beta_initial = 2.0       # Initial projection sharpness
eta_val = 0.5            # Projection threshold
max_iter = 200           # Maximum MMA iterations per phase

# Mesh Definition
mesh = RectangleMesh(nx, ny, Lx, Ly, quadrilateral=True)

# Function Spaces
V_u = VectorFunctionSpace(mesh, "CG", 1)  # Displacement
V_ctrl = FunctionSpace(mesh, "DG", 0)     # Control density
V_filt = FunctionSpace(mesh, "CG", 1)     # Filtered density
cell_volume = CellVolume(mesh)

x_ctrl = Function(V_ctrl, name="ControlDensity")

# --- NON-DESIGN REGION DEFINITION ---
nd_thickness = 0.05  # Thickness of the solid layer at the top boundary
dg_coords = Function(VectorFunctionSpace(mesh, "DG", 0)).interpolate(SpatialCoordinate(mesh)).dat.data_ro
nd_indices = np.where(dg_coords[:, 1] >= (Ly - nd_thickness))[0]
design_indices = np.setdiff1d(np.arange(V_ctrl.dim()), nd_indices)

x_tilde = Function(V_filt, name="FilteredDensity")
x_bar = Function(V_ctrl, name="ProjectedDensity")
vm_stress = Function(V_ctrl, name="vonMisesStress")
u_sol = Function(V_u, name="Displacement")

bc = [DirichletBC(V_u, Constant((0, 0)), 1), 
      DirichletBC(V_u, Constant((0, 0)), 2)]

x = SpatialCoordinate(mesh)
t_top = as_vector([0.0, traction_y])

# --- DYNAMIC CONSTANTS FOR MULTIPLE PHASES ---
delta_T = Constant(0.0)  # Controls thermal expansion
c_pen = Constant(0.0)    # Controls stress penalty
alpha = Constant(alpha_val)
sigma_y = Constant(sigma_yield_val)

## 3. The Thermoelastic Forward Problem

We use the **SIMP (Solid Isotropic Material with Penalization)** method to map our density field to the material's stiffness. 
We also define a Helmholtz filter and a continuous Heaviside projection to ensure the optimized designs are smooth and manufacture-ready (free of checkerboarding and binary).

In [ ]:
E = E_min + x_bar**Constant(p_mat) * (E_max - E_min)
lmbda = (E * nu) / ((1 + nu) * (1 - 2 * nu))
mu = E / (2 * (1 + nu))

def epsilon(u):
    return 0.5 * (grad(u) + grad(u).T)

def sigma(u, dT):
    thermal_stress = (3 * lmbda + 2 * mu) * alpha * dT
    return lmbda * div(u) * Identity(2) + 2 * mu * epsilon(u) - thermal_stress * Identity(2)

u_trial = TrialFunction(V_u)
v_test = TestFunction(V_u)

elastic_form = inner(sigma(u_trial, delta_T), epsilon(v_test)) * dx - dot(t_top, v_test) * ds(4)
a_form = lhs(elastic_form)
L_form = rhs(elastic_form)

problem = LinearVariationalProblem(a_form, L_form, u_sol, bcs=bc)
solver = LinearVariationalSolver(problem)

# Filter setup
r_min = Constant(r_min_val)
beta_proj = Constant(beta_initial)
eta_proj = Constant(eta_val)

p_filt, q_filt = TrialFunction(V_filt), TestFunction(V_filt)
a_h = (r_min**2 * inner(grad(p_filt), grad(q_filt)) + inner(p_filt, q_filt)) * dx
L_h = inner(x_ctrl, q_filt) * dx
h_solver = LinearVariationalSolver(LinearVariationalProblem(a_h, L_h, x_tilde))

def projection_filter(x_tilde, beta, eta):
    return (tanh(beta * eta) + tanh(beta * (x_tilde - eta))) / (tanh(beta * eta) + tanh(beta * (1 - eta)))

# STRESS POST-PROCESSING & PENALTY FUNCTION
s_tensor = sigma(u_sol, delta_T)
vm_expr = sqrt(s_tensor[0,0]**2 + s_tensor[1,1]**2 - s_tensor[0,0]*s_tensor[1,1] + 3*s_tensor[0,1]**2 + 1e-8)
stress_violation = max_value(0.0, vm_expr / sigma_y - 1.0)

control = Control(x_ctrl)
total_vol = assemble(Constant(1.0) * dx(mesh))
grad_J, grad_C = Function(V_ctrl), Function(V_ctrl)

## 4. Material Interpolation (SIMP)

In Topology Optimisation, we need a mathematical way to link our design variable (the density $x$) to the physical properties of the material (like the Young's modulus $E$). The most common approach is the **SIMP** (Solid Isotropic Material with Penalisation) method.

The SIMP interpolation is defined as:
$$E(x) = E_{\min} + x^p (E_{\max} - E_{\min})$$

Where:
* $x$ is the continuous density variable between **0** and **1**.
* $E_{\min}$ is a very small stiffness assigned to void regions (to prevent the stiffness matrix from becoming singular).
* $E_{\max}$ is the stiffness of the solid material.
* $p$ is the penalisation power (usually $p \ge 3$).

**Why use $p > 1$?** It penalises intermediate "grey" densities by making them disproportionately weak compared to their volume. This steers the optimiser toward a purely binary black-and-white design. Let's visualise this below.

In [ ]:
# Visualize the SIMP interpolation
x_vals = np.linspace(0, 1, 100)
p_values = [1, 3, 5]

plt.figure(figsize=(5, 5))  # Square plot
for p in p_values:
    E_vals = E_min + x_vals**p * (E_max - E_min)
    plt.plot(x_vals, E_vals, label=f'p = {p}')

plt.title("SIMP Material Interpolation")
plt.xlabel("Density (x)")
plt.ylabel("Young's Modulus (E)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Understanding Pseudo Density and Filters

Before running the optimisation, it is helpful to visualise what the control variables and filters are actually doing to our design. 

In Topology Optimisation, the **pseudo density** (`x_ctrl`) is a continuous mathematical field ranging from **0** (void) to **1** (solid). However, if we feed this raw density directly into the physics solver, the optimiser often exploits the finite element mesh, creating unmanufacturable "checkerboard" patterns.

To solve this, we use a two-step regularisation process:

1. **The Helmholtz Filter:** We apply a PDE-based density filter to the raw pseudo density to smooth it out spatially. This prevents checkerboarding but leaves us with a blurry, semi-solid grey area at the design boundaries. The filter is defined by the following Helmholtz partial differential equation:
$$-r_{\min}^2 \nabla^2 \tilde{x} + \tilde{x} = x$$
Where:
* $x$ is the raw design variable (pseudo density).
* $\tilde{x}$ is the resulting filtered density.
* $r_{\min}$ is the filter radius that controls the length scale of the smoothing.

2. **The Heaviside Projection:** To recover a crisp, binary design (mostly **0**s and **1**s), we apply a continuous approximation of the Heaviside step function to the filtered density. It is formulated using hyperbolic tangents to ensure it remains fully differentiable for the optimiser:
$$\bar{x} = \frac{\tanh(\beta \eta) + \tanh(\beta(\tilde{x} - \eta))}{\tanh(\beta \eta) + \tanh(\beta(1 - \eta))}$$
Where:
* $\bar{x}$ is the final projected density that is mapped to the material properties.
* $\beta$ is the penalisation parameter that controls the steepness or "sharpness" of the projection.
* $\eta$ is the projection threshold (typically set to 0.5).

Below, we define a raw pseudo density with artificial vertical bands, and then systematically apply different $r_{\min}$ and $\beta$ values to see their physical effects.

In [ ]:
# --- PLOTTING HELPER FOR VISUALIZATION ---
def plot_continuous_density_ax(density, ax, fig, title):
    cont = tripcolor(density, axes=ax, cmap='Greys', vmin=0.0, vmax=1.0)
    fig.colorbar(cont, ax=ax, label='Density', fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')

# --- 1. DEFINE RAW PSEUDO DENSITY (VERTICAL BANDS) ---
print("--- 1. Raw Pseudo Density ---")
x_coords = SpatialCoordinate(mesh)[0]
band_expr = conditional(sin(2 * pi * x_coords / 0.6) > 0, 1.0, 0.0)
x_ctrl.interpolate(band_expr)

fig, ax = plt.subplots(figsize=(6, 3))
plot_continuous_density_ax(x_ctrl, ax, fig, "Raw Pseudo Density (x_ctrl)")
plt.tight_layout()
plt.show()

# --- 2. APPLY HELMHOLTZ FILTER WITH DIFFERENT r_min ---
print("--- 2. Effect of the Helmholtz Filter (r_min) ---")
r_tests = [0.05, 0.15]
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, r_test in zip(axes, r_tests):
    r_min.assign(r_test)
    h_solver.solve()
    plot_continuous_density_ax(x_tilde, ax, fig, f"Filtered Density (r_min = {r_test})")
plt.tight_layout()
plt.show()

# --- 3. APPLY HEAVISIDE PROJECTION WITH DIFFERENT beta ---
print("--- 3. Effect of the Heaviside Projection (beta) ---")
r_min.assign(0.08)
h_solver.solve()

b_tests = [1.0, 8.0, 64.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, b_test in zip(axes, b_tests):
    x_bar.project(projection_filter(x_tilde, Constant(b_test), eta_proj))
    plot_continuous_density_ax(x_bar, ax, fig, f"Projected Density (beta = {b_test})")
plt.tight_layout()
plt.show()

# --- CLEANUP ---
r_min.assign(r_min_val)
x_ctrl.assign(Constant(vol_frac))
beta_proj.assign(beta_initial)


## 6. The MMA Optimisation Loop

Here we wrap our optimisation algorithm into a function `run_optimization`. We use the `mode` flag to seamlessly switch between minimising **compliance** (with a volume constraint) and minimising **volume** (with a compliance constraint). 

The respective optimisation formulations are mathematically defined as:

**Mode 1: Compliance Minimisation**
* **Objective:** $\min_{x} (J_{comp} + J_{stress})$
* **Constraint:** $V(x) \le V_{target}$

**Mode 2: Volume Minimisation**
* **Objective:** $\min_{x} (V(x) + J_{stress})$
* **Constraint:** $J_{comp} \le C_{max}$


In [ ]:
def run_optimization(iters, mode="compliance"):
    # Reset from uniform density every time to ensure an independent run
    init_val = vol_frac if mode == "compliance" else 1.0
    x_ctrl.assign(Constant(init_val))
    x_ctrl.dat.data[nd_indices] = 1.0 
    beta_proj.assign(beta_initial)
    get_working_tape().clear_tape()

    # Define n as the number of designable variables only
    n_total = V_ctrl.dim()
    n, m = len(design_indices), 1
    xval = np.ones((n, 1)) * init_val
    xmin, xmax = np.ones((n, 1)) * 1e-3, np.ones((n, 1)) * 1.0
    
    obj_history = []
    max_stress_history = []
    comp_history = []
    vol_history = []

    xold1, xold2 = xval.copy(), xval.copy()
    low, upp = xmin.copy(), xmax.copy()
    a0, epsimin = 1.0, 1e-7
    a, c, d = np.zeros((m, 1)), np.ones((m, 1)) * 5000.0, np.ones((m, 1))
    raa0, raa0eps = 0.01, 1e-6
    raa, raaeps = np.ones((m, 1)) * 0.01, np.ones((m, 1)) * 1e-6

    J_comp_scale = 1.0

    print("="*80)
    print(f" Starting {mode.capitalize()} Opt | delta_T = {float(delta_T)} | c_pen = {float(c_pen)} ".center(80, "="))
    print("="*80)
    print(f"{'Iter':>4} | {'Obj_f0':>12} | {'Stress':>12} | {'Constr_f1':>12} | {'Beta':>4} | {'Penalty':>8}")
    print("-" * 80)

    for i in range(1, iters + 1):
        x_ctrl.dat.data[design_indices] = xval.flatten()

        continue_annotation()
        h_solver.solve()
        x_bar.project(projection_filter(x_tilde, beta_proj, eta_proj))
        solver.solve()
        
        J_comp_raw = ((assemble(dot(t_top, u_sol) * ds(4)))**2 + 1e-8)**(1/2)
        J_stress_raw = assemble(x_bar * c_pen * stress_violation**2 * dx) / total_vol
        Vol_raw = assemble(x_bar * dx) / total_vol
        
        if i == 1:
            J_comp_scale = 1.0 / (float(J_comp_raw) + 1e-8)
        
        if mode == "compliance":
            J_obj = J_comp_raw * J_comp_scale + J_stress_raw
            C_constr = Vol_raw - vol_frac
        else:
            J_obj = Vol_raw + J_stress_raw
            C_constr = (J_comp_raw * J_comp_scale) - 1.2

        dJdx = compute_gradient(J_obj, control)
        dCdx = compute_gradient(C_constr, control)
        
        grad_J.project(dJdx * cell_volume)
        grad_C.project(dCdx * cell_volume)

        vm_stress.project(vm_expr)
        obj_history.append(float(J_obj))
        max_stress_history.append(float(vm_stress.dat.data_ro.max()))
        comp_history.append(float(J_comp_raw))
        vol_history.append(float(Vol_raw))

        pause_annotation()
        get_working_tape().clear_tape()

        f0val, df0dx = np.array([[float(J_obj)]]), grad_J.dat.data_ro[design_indices].reshape(n, 1)
        fval, dfdx = np.array([[float(C_constr)]]), grad_C.dat.data_ro[design_indices].reshape(m, n)

        low, upp, raa0, raa = asymp(i, n, xval, xold1, xold2, xmin, xmax, low, upp, raa0, raa, raa0eps, raaeps, df0dx, dfdx)
        xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx, fval, dfdx, a0, a, c, d)

        innerit = 0
        while innerit < 5:
            x_ctrl.dat.data[design_indices] = xmma.flatten()
            h_solver.solve()
            x_bar.project(projection_filter(x_tilde, beta_proj, eta_proj))
            solver.solve()
            
            J_comp_new_raw = ((assemble(dot(t_top, u_sol) * ds(4)))**2 + 1e-8)**(1/2)
            J_stress_new_raw = assemble(x_bar * c_pen * stress_violation**2 * dx) / total_vol
            Vol_new_raw = assemble(x_bar * dx) / total_vol

            if mode == "compliance":
                J_new = J_comp_new_raw * J_comp_scale + J_stress_new_raw
                C_new = Vol_new_raw - vol_frac
            else:
                J_new = Vol_new_raw + J_stress_new_raw
                C_new = (J_comp_new_raw * J_comp_scale) - 1.2
                
            f0valnew, fvalnew = np.array([[float(J_new)]]), np.array([[float(C_new)]])

            if concheck(m, epsimin, f0app, f0valnew, fapp, fvalnew):
                break
            innerit += 1
            raa0, raa = raaupdate(xmma, xval, xmin, xmax, low, upp, f0valnew, fvalnew, f0app, fapp, raa0, raa, raa0eps, raaeps, epsimin)
            xmma, _, _, _, _, _, _, _, _, f0app, fapp = gcmmasub(m, n, i, epsimin, xval, xmin, xmax, low, upp, raa0, raa, f0val, df0dx, fval, dfdx, a0, a, c, d)

        xold2, xold1, xval = xold1.copy(), xval.copy(), xmma.copy()

        if i % 30 == 0:
            beta_proj.assign(min(float(beta_proj) * 2.0, 32.0))
            if float(c_pen) > 0:
                c_pen.assign(float(c_pen) * 2.0)

        if i % 20 == 0 or i == 1:
            print(f"{i:4d} | {float(J_obj):12.4e} | {float(J_stress_raw):12.4e} | {float(C_constr):12.4e} | {float(beta_proj):4.1f} | {float(c_pen):8.1f}")

    print("="*80)
    
    x_res = Function(V_ctrl).assign(x_bar)
    s_res = Function(V_ctrl).assign(vm_stress)
    return x_res, s_res, obj_history, max_stress_history, comp_history, vol_history

def plot_topology(density, title):
    fig, ax = plt.subplots(figsize=(10, 4))
    v_bw = Function(density.function_space())
    v_bw.dat.data[:] = np.where(density.dat.data_ro >= 0.3, 1.0, 0.0)
    cont = tripcolor(v_bw, axes=ax, cmap='Greys')
    fig.colorbar(cont, ax=ax, label='Material')
    ax.set_title(f"{title} - Binary Design")
    ax.set_aspect('equal')
    ax.axis('off')
    plt.show()

def plot_stress_map(stress, title):
    fig, ax = plt.subplots(figsize=(10, 4))
    cont = tripcolor(stress, axes=ax, cmap='turbo')
    fig.colorbar(cont, ax=ax, label='von Mises Stress')
    ax.set_title(f"{title} - Stress Distribution")
    ax.set_aspect('equal')
    ax.axis('off')
    plt.show()

def plot_log_metric(metric_hist, ylabel, title):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(metric_hist, color='tab:blue', label=ylabel)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(ylabel)
    ax.set_yscale('log')
    ax.legend()
    plt.title(title)
    plt.grid(True, which='both', ls='-', alpha=0.3)
    plt.show()

## 7. Phase 1: Purely Elastic Topology Optimisation

We start with a standard TO problem. $\Delta T$ is set to **0** and the stress penalty is turned off. 

**Optimisation Problem Formulation:**
**Minimise (Mechanical Compliance):**
$$ \min_{\bar{x}} \quad \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{u} \, ds $$

**Subject to (Volume Constraint & Elastic Equilibrium):**
$$ \frac{1}{|\Omega|} \int_{\Omega} \bar{x} \, dx \le V_{\text{target}} $$
$$ \int_{\Omega} \sigma(\mathbf{u}) : \varepsilon(\mathbf{v}) \, dx = \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{v} \, ds \quad \forall \mathbf{v} \in V $$

**Takeaway 1:** In a purely mechanical scenario, minimising compliance generally leads to reasonable stress levels. A stiffer structure naturally distributes the mechanical load more efficiently, avoiding massive stress concentrations.

In [ ]:
delta_T.assign(0.0)
c_pen.assign(0.0)
x_elastic, s_elastic, h_e, st_e, comp_e, vol_e = run_optimization(max_iter)

plot_topology(x_elastic, "Elastic Only")
plot_stress_map(s_elastic, "Elastic Only")
plot_log_metric(comp_e, "Absolute Compliance", "Phase 1: Absolute Compliance")

## 8. Phase 2: Thermoelastic TO (Compliance Objective)

Now we apply a **50°C** thermal load. However, our objective remains strictly to minimise compliance (maximise stiffness).

**Optimisation Problem Formulation:**
**Minimise (Mechanical Compliance):**
$$ \min_{\bar{x}} \quad \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{u} \, ds $$

**Subject to (Volume Constraint & Thermoelastic Equilibrium):**
$$ \frac{1}{|\Omega|} \int_{\Omega} \bar{x} \, dx \le V_{\text{target}} $$
$$ \int_{\Omega} \sigma(\mathbf{u}, \Delta T) : \varepsilon(\mathbf{v}) \, dx = \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{v} \, ds \quad \forall \mathbf{v} \in V $$

**Takeaway 2:** When optimising for compliance under combined loads, the optimiser will often place material in a way that uses the thermal expansion to actively "push back" against the mechanical load. 

In [ ]:
delta_T.assign(delta_T_target)
c_pen.assign(0.0)
x_thermo, s_thermo, h_t, st_t, comp_t, vol_t = run_optimization(max_iter)

plot_topology(x_thermo, f"Thermoelastic (dT = {delta_T_target}, Pen = 0)")
plot_stress_map(s_thermo, f"Thermoelastic (dT = {delta_T_target}, Pen = 0)")
plot_log_metric(comp_t, "Absolute Compliance", f"Phase 2: Absolute Compliance (dT = {delta_T_target})")

## 9. Phase 3: Thermoelastic TO with Stress Penalty

To solve the high-stress issue, we introduce a quadratic stress penalty term to the objective function. The optimiser is now punished if the von Mises stress exceeds our defined yield limit ($\sigma_y$ = **80**).

**Optimisation Problem Formulation:**
**Minimise (Compliance + Stress Penalty):**
$$ \min_{\bar{x}} \quad \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{u} \, ds + \frac{c_{\text{pen}}}{|\Omega|} \int_{\Omega} \bar{x} \max\left(0, \frac{\sigma_{VM}}{\sigma_y} - 1\right)^2 \, dx $$

**Subject to (Volume Constraint & Thermoelastic Equilibrium):**
$$ \frac{1}{|\Omega|} \int_{\Omega} \bar{x} \, dx \le V_{\text{target}} $$
$$ \int_{\Omega} \sigma(\mathbf{u}, \Delta T) : \varepsilon(\mathbf{v}) \, dx = \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{v} \, ds \quad \forall \mathbf{v} \in V $$

**Takeaway 3:** Explicitly accounting for stress yields a much safer, practical design. 

However, maintaining a strict *volume fraction constraint* (e.g., exactly **30%** volume) becomes fundamentally problematic in thermoelasticity. The thermal load is proportional to the amount of material present. By forcing the optimiser to use exactly **30%** of the volume, we might be forcing it to retain material that only serves to generate more thermal stress. Thus, the volume constraint often becomes active in a harmful way.

In [ ]:
delta_T.assign(delta_T_target)
c_pen.assign(c_penalty_val)
x_stress, s_stress, h_s, st_s, comp_s, vol_s = run_optimization(max_iter)

plot_topology(x_stress, f"Stress Constrained (dT={delta_T_target}, Pen={c_penalty_val})")
plot_stress_map(s_stress, f"Stress Constrained (dT={delta_T_target}, Pen={c_penalty_val})")
plot_log_metric(comp_s, "Absolute Compliance", f"Phase 3: Absolute Compliance (dT={delta_T_target})")

## 10. Phase 4: Volume Minimisation under Constraints

A much more physically meaningful approach to thermoelastic design is to switch our objective completely. Instead of minimising compliance for a fixed volume, we **minimise volume** while ensuring that both the mechanical compliance and the maximum stress stay below strict limits.

**Optimisation Problem Formulation:**
**Minimise (Volume + Stress Penalty):**
$$ \min_{\bar{x}} \quad \frac{1}{|\Omega|} \int_{\Omega} \bar{x} \, dx + \frac{c_{\text{pen}}}{|\Omega|} \int_{\Omega} \bar{x} \max\left(0, \frac{\sigma_{VM}}{\sigma_y} - 1\right)^2 \, dx $$

**Subject to (Compliance Constraint & Thermoelastic Equilibrium):**
$$ \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{u} \, ds \le C_{\max} $$
$$ \int_{\Omega} \sigma(\mathbf{u}, \Delta T) : \varepsilon(\mathbf{v}) \, dx = \int_{\Gamma_{\text{top}}} \mathbf{t} \cdot \mathbf{v} \, ds \quad \forall \mathbf{v} \in V $$

*Where $C_{\max}$ is the maximum allowable mechanical compliance.*

**Takeaway 4:** Volume minimisation allows the optimiser to remove material that is actively contributing to thermal stress buildup, provided the remaining structure is stiff enough to handle the mechanical loads. This results in the most elegant and efficient thermoelastic designs.

In [ ]:
delta_T.assign(delta_T_target)
c_pen.assign(c_penalty_val)
# Switching mode to "volume" will enforce a compliance constraint instead of a volume constraint
x_vol, s_vol, h_v, st_v, comp_v, vol_v = run_optimization(max_iter, mode="volume")

plot_topology(x_vol, f"Volume Min (dT={delta_T_target}, Pen={c_penalty_val})")
plot_stress_map(s_vol, f"Volume Min (dT={delta_T_target}, Pen={c_penalty_val})")
plot_log_metric(vol_v, "Volume Fraction", f"Phase 4: Absolute Volume Objective (dT={delta_T_target})")

## 11. Convergence and Summary

Let's plot the convergence histories. Pay close attention to the stress history graph:
* Notice how Phase 1 (Elastic) remains at relatively low stress.
* Phase 2 (Thermoelastic without penalty) shoots way above the yield limit.
* Phase 3 successfully suppresses the stress to stay below the yield threshold line.

In [ ]:
def plot_compliance_comparison(histories, labels):
    plt.figure(figsize=(10, 5))
    for hist, label in zip(histories, labels):
        plt.plot(hist, label=label)
    plt.yscale('log')
    plt.xlabel('Iteration')
    plt.ylabel('Absolute Compliance')
    plt.title("Absolute Compliance Comparison (Compliance-Based Phases)")
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.5)
    plt.show()

def plot_convergence_stress(stress_histories, labels):
    plt.figure(figsize=(10, 5))
    for s_hist, label in zip(stress_histories, labels):
        plt.plot(s_hist, '--', label=label)
    plt.axhline(y=sigma_yield_val, color='k', linestyle=':', label="Yield Limit")
    plt.xlabel('Iteration')
    plt.ylabel('Max von Mises Stress')
    plt.title("Max Stress History (Compliance-Based Phases)")
    plt.legend()
    plt.grid(True, which="both", ls="-", alpha=0.5)
    plt.show()

labels = ["Elastic", "Thermo", "Stress-Constrained"]
plot_compliance_comparison([comp_e, comp_t, comp_s], labels)
plot_convergence_stress([st_e, st_t, st_s], labels)